# Tugas Praktisi Mengajar: Make Sense of Data with Analysis and AI
**Nama:** (Ayuni Salsyabilla Putri)  
**NPM:** (G1A024096)  

## Studi Kasus: Prediksi Keterlambatan Pembayaran (Late Payment) PT Maju Bersama
Studi kasus ini menerapkan siklus **CRISP-DM** untuk memprediksi *invoice* mana yang berpotensi mengalami keterlambatan pembayaran berdasarkan tipe pelanggan, total tagihan, dan batas waktu hari.

### Tahapan yang Dilakukan:
1. **Data Understanding & Preparation:** Menyiapkan data sintetis dan mengubah nilai teks (Kategori) menjadi angka menggunakan *Label Encoder* agar bisa dibaca oleh *Machine Learning*.
2. **Mencegah Data Leakage:** Membuang kolom prediksi status berupa teks agar model AI tidak "menyontek" hasil akhir sebelum belajar.
3. **Modeling:** Menggunakan dan membandingkan 3 algoritma sekaligus (Logistic Regression, Decision Tree, Random Forest) layaknya analis data profesional.
4. **Evaluation:** Mengevaluasi metrik kecerdasan (*Accuracy, Precision, Recall*) untuk menentukan algoritma mana yang paling akurat memecahkan masalah bisnis.

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta, date
import os

np.random.seed(42)
jumlah_data = 1000

data = {
    'ID_Invoice': [f'INV-{i:04d}' for i in range(1, jumlah_data + 1)],
    'Tipe_Pelanggan': np.random.choice(['Retail', 'Corporate', 'Government'], jumlah_data, p=[0.5, 0.4, 0.1]),
    'Total_Tagihan': np.random.uniform(1000000, 50000000, jumlah_data).round(-4),
    'Batas_Waktu_Hari': np.random.choice([15, 30, 45], jumlah_data),
}

df = pd.DataFrame(data)

prob_telat = np.where(df['Tipe_Pelanggan'] == 'Corporate', 0.4, 0.15)
is_late = np.random.binomial(1, prob_telat)

df['Status_Pembayaran'] = np.where(is_late == 1, 'Terlambat', 'Tepat Waktu')

os.makedirs('../dataset', exist_ok=True)
df.to_csv('../dataset/data_invoice.csv', index=False)
print("Dataset data_invoice.csv telah dibuat di dalam folder dataset.")

import warnings
warnings.filterwarnings('ignore')

Dataset data_invoice.csv telah dibuat di dalam folder dataset.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv('../dataset/data_invoice.csv')

le = LabelEncoder()
df['Tipe_Pelanggan_Angka'] = le.fit_transform(df['Tipe_Pelanggan'])

df['Status_Angka'] = df['Status_Pembayaran'].apply(lambda x: 1 if x == 'Terlambat' else 0)

X = df[['Tipe_Pelanggan_Angka', 'Total_Tagihan', 'Batas_Waktu_Hari']]
y = df['Status_Angka']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)          
    y_pred = model.predict(X_test)       
    
    print(f"========== Hasil Model: {name} ==========")
    print(f"Akurasi Total: {accuracy_score(y_test, y_pred):.2f}")
    print(classification_report(y_test, y_pred))
    print("\n")

========== Hasil Model: Logistic Regression ==========
Akurasi Total: 0.73
              precision    recall  f1-score   support

           0       0.73      1.00      0.84       219
           1       0.00      0.00      0.00        81

    accuracy                           0.73       300
   macro avg       0.36      0.50      0.42       300
weighted avg       0.53      0.73      0.62       300



========== Hasil Model: Decision Tree ==========
Akurasi Total: 0.67
              precision    recall  f1-score   support

           0       0.76      0.80      0.78       219
           1       0.37      0.32      0.34        81

    accuracy                           0.67       300
   macro avg       0.57      0.56      0.56       300
weighted avg       0.66      0.67      0.66       300



========== Hasil Model: Random Forest ==========
Akurasi Total: 0.68
              precision    recall  f1-score   support

           0       0.77      0.81      0.79       219
           1       0